# Review the end-to-end project

Start with the business decision: after 24 hours without a purchase, which shoppers return during the next seven days? Prediction performance is observed. Coupon effects and financial assumptions are not observed.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Markdown
ROOT = Path.cwd()
if not (ROOT / "reports" / "model_summary.json").exists():
    ROOT = ROOT.parent
assert (ROOT / "reports" / "model_summary.json").exists(), "Run the pipeline first."
REPORTS = ROOT / "reports"

## 1 Check the source and eligible population

Source events, cleaned core events and eligible journeys are different counts. Page visits and searches are not model features.

In [2]:
audit = json.loads((REPORTS / "pipeline_audit.json").read_text())
print("Source events:", format(audit["source_events"], ","))
print("Cleaned core events:", format(audit["cleaned_core_events"], ","))
print("Eligible journeys:", format(audit["counts"]["eligible"], ","))
display(pd.read_csv(REPORTS / "split_summary.csv"))

Source events: 225,224,262
Cleaned core events: 11,852,529
Eligible journeys: 2,959,441


,split,journeys,purchasers,purchase_rate,first_decision,last_decision
0,train,880660,15814.0,0.017957,2022-07-24 00:12:35,2022-09-07 23:59:50
1,purged,304111,7758.0,0.025510,2022-09-08 00:00:05,2022-10-14 23:58:10
2,validation,494175,12485.0,0.025264,2022-09-15 00:00:15,2022-10-07 23:59:50
3,test,1280495,31349.0,0.024482,2022-10-15 00:00:20,2022-12-01 00:09:15


## 2 Interpret the later-period test

Average precision measures ranking. A constant baseline has average precision equal to outcome prevalence. ROC-AUC, calibration, Brier score and log loss provide additional evidence. Top-20% purchase capture describes the highest-scoring group; the simulated selective coupon policy uses the lowest-scoring group. These are different uses of the ranking.

In [3]:
display(pd.read_csv(REPORTS / "validation_metrics.csv"))
display(pd.read_csv(REPORTS / "test_metrics.csv"))
display(pd.read_csv(REPORTS / "calibration.csv"))

,model,average_precision,roc_auc,brier_score,log_loss,top20_purchase_capture
0,logistic_regression,0.055421,0.700373,0.024388,0.112466,0.453825
1,gradient_boosting,0.067810,0.730558,0.024166,0.109825,0.501241


,model,average_precision,roc_auc,brier_score,log_loss,top20_purchase_capture
0,constant_baseline,0.024482,0.500000,0.023926,0.116124,0.200000
1,gradient_boosting,0.068134,0.735129,0.023388,0.106351,0.508023


,predicted_probability,observed_purchase_rate
0,0.005508,0.005063
1,0.007347,0.007172
2,0.008890,0.008522
3,0.010423,0.010417
4,0.012774,0.013666
5,0.015464,0.017417
6,0.019213,0.022367
7,0.028114,0.035845
8,0.046844,0.048435
9,0.083513,0.075940


## 3 Understand shopper behavior

Compare rates within each dimension. These associations do not show that previous purchases or repeated carts cause future purchases. Permutation importance is calculated on a validation-only sample, not the final test set.

In [4]:
display(pd.read_csv(REPORTS / "behavior_segments.csv"))
display(pd.read_csv(REPORTS / "validation_importance.csv"))

,dimension,segment,journeys,purchasers,purchase_rate
0,Cart activity in first 24 hours,One cart event,1180789,27603.0,0.023377
1,Cart activity in first 24 hours,Repeated cart events,99706,3746.0,0.037570
2,Observed purchase recency,8-30 days,103878,6429.0,0.061890
3,Observed purchase recency,No observed prior purchase,928711,12640.0,0.013610
4,Observed purchase recency,Over 30 days,107173,5646.0,0.052681
5,Observed purchase recency,Within 7 days,140733,6634.0,0.047139
6,Prior observed purchases,0 purchases,928711,12640.0,0.013610
7,Prior observed purchases,1-2 purchases,155831,6620.0,0.042482
8,Prior observed purchases,3+ purchases,195953,12089.0,0.061693


,feature,average_precision_decrease,repeat_std
0,user_buy_events,0.020447,0.001663
1,days_since_buy,0.010293,0.001543
2,user_cart_events,0.010224,0.001475
3,observed_history_days,0.007531,0.000822
4,product_cart_events_24h,0.004175,0.000704
5,sku_cart_events,0.001798,0.000767
6,sku_buy_events,0.001252,0.000789
7,decision_dow,0.000370,0.000289
8,decision_hour,-0.000754,0.000845


## 4 Challenge the economics

Financial outputs are simulations. Start with zero response, then examine 1% and 3%. Compare selective targeting with random targeting at equal contacts and with blanket offers at a different contact count. A higher total contribution with blanket offers can be rational when assumed incremental response is high enough.

In [5]:
scenarios = pd.read_csv(REPORTS / "central_scenarios.csv")
display(scenarios[["assumed_response", "policy", "contacts", "contribution_per_1000", "incremental_per_1000"]])
display(Markdown((REPORTS / "EXECUTIVE_SUMMARY.md").read_text()))

,assumed_response,policy,contacts,contribution_per_1000,incremental_per_1000
0,0.00,no_new_offers,0,856.867852,0.000000
1,0.00,blanket,1280495,562.048466,-294.819386
2,0.00,random_20pct,256099,797.285464,-59.582388
3,0.00,selective_20pct,256099,834.653825,-22.214027
4,0.01,no_new_offers,0,856.867852,0.000000
5,0.01,blanket,1280495,805.927981,-50.939871
6,0.01,random_20pct,256099,846.045904,-10.821948
7,0.01,selective_20pct,256099,884.348475,27.480623
8,0.03,no_new_offers,0,856.867852,0.000000
9,0.03,blanket,1280495,1293.687012,436.819160


# Discount targeting analysis

## Main finding

The selected gradient boosting model achieved test average precision of 0.0681, versus 0.0245 for a constant baseline (+178.3% relative difference). The observed test purchase rate was 2.45% across 1,280,495 eligible journeys. These are purchase-prediction results, not measured discount effects.

## Data and scope

The source contains 225,224,262 behavioral events. The journey pipeline directly cleans 11,852,529 cart, purchase and removal events and produces 2,959,441 eligible first shopper-product journeys. Page visits and searches are inventoried but not model inputs. Activity is from 2022-06-23 00:12:25 through 2022-12-08 00:09:15; the dataset was released for the 2025 challenge. A 30-day warm-up and full seven-day outcome window are required. Exact duplicate events are collapsed.

## Financial scenario

At an assumed $100 order value, 35% contribution margin, 10% discount, $0.05 contact cost, and a uniform 3% coupon response among otherwise nonbuyers, selective offers differ from blanket offers by $-309.95 in simulated contribution per 1,000 eligible journeys. This assumption-driven comparison changes the number of contacts. The random-20% policy provides an equal-contact comparator. The dashboard varies response, margins, discounts and the selected group's relative responsiveness, including zero-response conditions.

This is not realized profit. Costs, actual prices, historical promotions, stockouts, quantities and incremental response are not observed. The model predicts return purchasing, not who a coupon persuades. No-new-offer outcomes are approximated using historical outcomes under unknown existing marketing.

## Recommendation

Use the purchase-ranking model to define strata for a randomized coupon experiment. Do not deploy the simulated best policy as a proven revenue intervention. Within each propensity band, compare coupon assignment with no new offer, using consent, customer-level assignment and contribution net of actual discounts and returns. Choose a minimum worthwhile effect and determine sample size from pilot variance. Monitor margin, opt-outs and repeated discount dependence.

## Reproduce

Run `.venv/Scripts/python.exe scripts/run_pipeline.py --stage all` from the repository root. Raw data, intermediate DuckDB tables, individual scores and model artifacts remain local. Published results are aggregates. See docs/METHODOLOGY.md for exact boundaries, exclusions, feature definitions and split dates.


## Next business step

Open reports/dashboard.html to vary the assumptions. Read docs/METHODOLOGY.md before presenting results. A real randomized coupon test is required to claim actual profit improvement.